# ***MOTOR***

In [1]:
# --- 0. IMPORTACIONES NECESARIAS ---
import re
import os
import ipaddress # Usaremos esto más
import socket
import json
import requests
import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
from urllib.parse import urlparse, unquote
from google.colab import files

# --- 1. FUNCIONES DE DESOFUSCACIÓN (OK) ---
def desofuscar_indicadores(texto):
    patrones = [
        (r'\[\.\]', '.'), (r'\[dot\]', '.'), (r'\((dot)\)', '.'),
        (r'\\[ \\]', ''), (r'\\[\\]', ''), (r'hxxps://', 'https://'),
        (r'hxxp://', 'http://'), (r'h\\*\\*p://', 'http://'), (r'htxp://', 'http://'),
        (r'\\[://\\]', '://'), (r'\\[:\\]', ':'), (r'//\\]', '//'),
    ]
    texto_desofuscado = texto
    for patron, reemplazo in patrones:
        texto_desofuscado = re.sub(patron, reemplazo, texto_desofuscado, flags=re.IGNORECASE)
    return texto_desofuscado

# --- 2. CLASE DE NORMALIZACIÓN (OK) ---
class IOCNormalizer:
    def normalize_ip(self, ip):
        try: return str(ipaddress.ip_address(ip.strip()))
        except ValueError: return None
    def normalize_domain(self, domain):
        domain = domain.strip().lower()
        if domain.startswith(('http://', 'https://')):
            parsed = urlparse(domain)
            domain = parsed.netloc or parsed.path
        domain = domain.split(':')[0]
        return domain if '.' in domain else None
    def normalize_url(self, url):
        try:
            url = url.strip()
            if not url.startswith(('http://', 'https://')): url = 'http://' + url
            parsed = urlparse(url)
            path = unquote(parsed.path)
            query = unquote(parsed.query) if parsed.query else ""
            normalized_url = f"{parsed.scheme}://{parsed.netloc}{path}"
            if query: normalized_url += f"?{query}"
            return normalized_url.lower()
        except: return None
    def normalize_ioc(self, ioc):
        ioc = ioc.strip()
        # El orden es importante: IP primero, luego URL, luego Dominio
        try:
            # Primero, intentar ver si es una IP válida
            ipaddress.ip_address(ioc)
            return self.normalize_ip(ioc)
        except ValueError:
            # Si no es IP, comprobar si es URL o Dominio
            if ioc.startswith(('http://', 'https://')) or '://' in ioc:
                return self.normalize_url(ioc)
            elif '.' in ioc:
                return self.normalize_domain(ioc)
        return ioc # Devolver tal cual si no es nada

# --- 3. CLASE DE CLASIFICACIÓN (CORREGIDA) ---
class IOCClassifier:
    def __init__(self):
        self.patterns = {
            # Regex de Dominio (no debe coincidir con una IP)
            'domain': r'^(?!(\d{1,3}\.){3}\d{1,3}$)([a-zA-Z0-9]([a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$',
            # Regex de URL (robusta)
            'url': r'^(https?|ftp)://[a-zA-Z0-9\-._~:/?#\[\]@!$&\'()*+,;=%]+(\.[a-zA-Z0-9\-._~:/?#\[\]@!$&\'()*+,;=%]+)+(/\S*)?$'
        }
    def classify_ioc(self, ioc):
        if not isinstance(ioc, str):
            return "Desconocido"
        ioc = ioc.strip()

        # El orden de clasificación es CRÍTICO

        # 1. ¿Es una URL? (La más específica)
        if re.match(self.patterns['url'], ioc, re.IGNORECASE):
            return "URL"

        # 2. ¿Es una IP? (Usando la librería ipaddress, no regex)
        try:
            ip = ipaddress.ip_address(ioc)
            if isinstance(ip, ipaddress.IPv4Address):
                return "IPv4"
            elif isinstance(ip, ipaddress.IPv6Address):
                return "IPv6"
        except ValueError:
            # No es una IP, continuar...
            pass

        # 3. ¿Es un Dominio?
        if re.match(self.patterns['domain'], ioc):
            return "Dominio"

        # 4. Si no es nada de lo anterior
        return "Desconocido"

# --- 4. FUNCIÓN DE RESOLUCIÓN DNS (OK) ---
def resolve_dns(ioc):
    results = {}
    if not isinstance(ioc, str):
        results['Error'] = 'Input no válido (None)'
        return results
    ioc = ioc.strip()
    socket.setdefaulttimeout(5)
    ioc_para_dns = ioc
    if ioc.startswith(('http://', 'https://')):
        try: ioc_para_dns = urlparse(ioc).hostname
        except:
            results['Error'] = 'Error al parsear la URL'
            return results
    if not ioc_para_dns:
        results['Error'] = 'No se pudo extraer hostname de la URL'
        return results
    try:
        ip = ipaddress.ip_address(ioc_para_dns)
        try:
            ptr_result = socket.gethostbyaddr(str(ip))
            results['PTR'] = ptr_result[0]
        except Exception: results['PTR'] = 'No se encontró registro PTR'
    except ValueError:
        try:
            results['A'] = socket.gethostbyname(ioc_para_dns)
        except Exception: results['A'] = 'No se encontró dirección IPv4'
        try:
            aaaa_records = socket.getaddrinfo(ioc_para_dns, None, socket.AF_INET6)
            results['AAAA'] = list(set([r[4][0] for r in aaaa_records])) or 'No se encontró dirección IPv6'
        except Exception: results['AAAA'] = 'No se encontró dirección IPv6'
    return results

# --- 5. FUNCIÓN DE INFORME RDAP (CORREGIDA) ---
def consultar_rdap(ioc):
    if not isinstance(ioc, str):
        return {"error": "Input no válido (None)"}
    try:
        ioc_limpia = ioc.strip()
        ioc_para_rdap = ioc_limpia
        tipo_consulta = "" # Para construir la URL de RDAP

        # Determinar el tipo de consulta
        if re.match(r'^(https?|ftp)://', ioc_limpia): # Si es URL
            ioc_para_rdap = urlparse(ioc_limpia).hostname
            tipo_consulta = "domain"
        else:
            try: # Intentar ver si es IP
                ip = ipaddress.ip_address(ioc_limpia)
                ioc_para_rdap = str(ip)
                tipo_consulta = "ip"
            except ValueError: # Asumir que es dominio
                ioc_para_rdap = ioc_limpia
                tipo_consulta = "domain"

        if not ioc_para_rdap:
            return {"error": "No se pudo extraer el IOC para la consulta"}

        # --- URL DE RDAP CORREGIDA ---
        # Usar el bootstrap genérico de rdap.org
        url = f"https://rdap.org/{tipo_consulta}/{ioc_para_rdap}"

        response = requests.get(url, headers={'Accept': 'application/rdap+json'}, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

# --- Instancias de las clases ---
normalizer = IOCNormalizer()
classifier = IOCClassifier()

print("✅ ¡Motor de análisis (v4 - Fix Clasificación y RDAP) cargado! Ejecuta la celda de la UI.")

✅ ¡Motor de análisis (v4 - Fix Clasificación y RDAP) cargado! Ejecuta la celda de la UI.


# ***UI***

In [2]:
# --- CREACIÓN DE LA INTERFAZ GRÁFICA (UI) ---

# 1. Crear los componentes
text_input = widgets.Text(
    value='',
    placeholder='Pega aquí tu IOC (ej: hxxp://dominio[.]com)',
    description='IOC:',
    layout=widgets.Layout(width='80%')
)

run_button = widgets.Button(
    description='Analizar IOC 🚀',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    icon='search'
)

# 3. Crear el área de salida
output_area = widgets.Output()

# 4. Definir la función que se ejecuta al pulsar el botón
def analizar_todo_en_cadena(b):
    # Limpiar salida anterior
    output_area.clear_output(wait=True)

    # Mostrar resultados dentro del área de salida
    with output_area:
        ioc_bruto = text_input.value
        if not ioc_bruto:
            print("❌ Error: Por favor, introduce un IOC para analizar.")
            return

        print(f"🔄 Analizando: {ioc_bruto}\n")

        # --- PASO 1: DESOFUSCAR ---
        print("="*20)
        print("🔍 1. DESOFUSCACIÓN")
        print("="*20)
        ioc_desofuscado = desofuscar_indicadores(ioc_bruto)
        print(f"   Original:    {ioc_bruto}")
        print(f"   Desofuscado: {ioc_desofuscado}\n")

        # --- PASO 2: NORMALIZAR ---
        print("="*20)
        print("🧹 2. NORMALIZACIÓN")
        print("="*20)
        ioc_normalizado = normalizer.normalize_ioc(ioc_desofuscado)
        print(f"   Normalizado: {ioc_normalizado}\n")

        # --- PASO 3: CLASIFICAR ---
        print("="*20)
        print("🏷️ 3. CLASIFICACIÓN")
        print("="*20)
        tipo_ioc = classifier.classify_ioc(ioc_normalizado)
        print(f"   Tipo: {tipo_ioc}\n")

        if tipo_ioc == "Desconocido":
            print("⚠️ El IOC es de tipo desconocido. No se puede continuar con el análisis de red (DNS/RDAP).")
            return

        # --- PASO 4: RESOLUCIÓN DNS ---
        print("="*20)
        print("🌐 4. RESOLUCIÓN DNS")
        print("="*20)
        print("   Consultando registros...")
        info_dns = resolve_dns(ioc_normalizado)
        # Usamos json.dumps para una salida bonita
        print(json.dumps(info_dns, indent=4))
        print("\n")

        # --- PASO 5: INFORME RDAP ---
        print("="*20)
        print("📜 5. INFORME RDAP")
        print("="*20)
        print("   Consultando información de registro...")
        info_rdap = consultar_rdap(ioc_normalizado)
        print(json.dumps(info_rdap, indent=4))
        print("\n")
        print("✅ Análisis completado.")

# 5. Conectar el botón a la función
run_button.on_click(analizar_todo_en_cadena)

# 6. Mostrar la UI
print("=== ANALIZADOR DE IOC INDIVIDUAL ===")
display(widgets.VBox([text_input, run_button, output_area]))

=== ANALIZADOR DE IOC INDIVIDUAL ===
